In [10]:
!pip install transformers datasets torch

In [11]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer, DataCollatorForLanguageModeling, Trainer, TrainingArguments
from torch.utils.data import Dataset

# Custom Dataset class (replaces TextDataset)
class TextDataset(Dataset):
    def __init__(self, tokenizer, file_path, block_size=128):
        with open(file_path, "r") as f:
            text = f.read()
        tokenized = tokenizer(text, return_tensors="pt", truncation=False)["input_ids"][0]
        self.examples = [tokenized[i:i+block_size] for i in range(0, len(tokenized) - block_size, block_size)]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return {"input_ids": self.examples[i], "labels": self.examples[i]}

# Load GPT-2
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Load dataset
train_dataset = TextDataset(tokenizer, "data.txt", block_size=128)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Training settings
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=500,
    logging_steps=100,
)
# Train
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
)

trainer.train()
print("✅ Training done!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Training done!


In [12]:
from transformers import pipeline

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

result = generator(
    "The secret to success is",
    max_new_tokens=80,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.9,
    pad_token_id=tokenizer.eos_token_id
)

print(result[0]['generated_text'])

[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The secret to success is its dedication.


In 2012 I took part in a survey of students in the community who had not received the National Medal for Leadership under President Obama. A big part of this survey was composed of students from around the world that had been told that they were in awe of one another, of their students, of these students, the experience of their students.

A lot of it was


In [13]:
prompts = [
    "Hard work and dedication will",
    "Believe in yourself because"
]

for p in prompts:
    result = generator(
        p,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
    print(f"Prompt: {p}")
    print(result[0]['generated_text'])
    print("---")

[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt: Hard work and dedication will
Hard work and dedication will give my son a strength that may not be matched by the things we do.

"The biggest pain is a kid's heart," the Rev. Dr. Daphne said.

He and his friends and family feel "that there is too much power in the past."

"The moment we all feel is the moment we all see the true joy of this universe," he
---
Prompt: Believe in yourself because
Believe in yourself because everyone else has already done so. You don't have to believe that every day they'll succeed, and that one day they'll be happy they do. It's worth every second."

"Success is just one of the many things that go on the journey. In the beginning all you have to do is try to do what you are trying to do. Don't only do what you heard
---
